In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/t1dbg

In [2]:
# verifica GPU
!nvidia-smi

In [ ]:
# Per installare rapids se non già disponibile
# !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
# !python rapidsai-csp-utils/colab/pip-install.py

In [3]:
import cuml
cuml.__version__

### Import librerie


In [4]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import pickle
import lightgbm as lgb
import xgboost as xgb

from cuml.ensemble import RandomForestRegressor

### Data Utilities


In [5]:
# Costanti
HYPO = 70.0
HYPER = 180.0
L_BOUND = 40.0
U_BOUND = 400.0

In [6]:
def load_splits(splits_dir="data/split_sets"):
    """Carica gli split e i metadati dai relativi file.
    Args:
        splits_dir (str): Directory contenente gli split set (default: 'data/split_sets')
    Returns:
        tuple: (train_set, val_set, test_set, X_cols, y_cols)
    """
    datasets = []
    for name in ["train", "val", "test"]:
        df = pd.read_parquet(f"{splits_dir}/{name}_set.parquet")
        datasets.append(df)

    with open(f"{splits_dir}/metadata.json", "r") as f:
        metadata = json.load(f)

    return tuple(datasets + [metadata["X_cols"], metadata["y_cols"]])

In [7]:
def rescale_data(df, rescale_cols):
    """Effettua il rescale dei dati back al loro intervallo originario.
    Args:
        df (pd.DataFrame): DataFrame con i dati da riscalare
        rescale_cols (list): List dei nomi delle colonne da riscalare
    Returns:
        pd.DataFrame: Dataframe con le colonne scelte riscalate
    """
    df = df.copy()
    for col in rescale_cols:
        df[col] = ((df[col] + 1) * (U_BOUND - L_BOUND) / 2) + L_BOUND
    return df

In [8]:
def calculate_metrics(df):
    """Calcola le metriche per ciascun paziente in un sottoinsieme.
    Args:
        df (pd.DataFrame): Dataframe con le colonne 'Patient_ID', 'target', e 'y_pred'
    Returns:
        tuple: (samples, maes, mapes, rmses) - numero di samples e lista delle metriche ottenute
    """
    samples = 0
    maes, mapes, rmses = [], [], []

    for patient_id in df["Patient_ID"].unique():
        patient_data = df[df["Patient_ID"] == patient_id]
        if patient_data.empty:
            continue

        samples += len(patient_data)
        maes.append(mean_absolute_error(patient_data["target"], patient_data["y_pred"]))
        mapes.append(
            mean_absolute_percentage_error(
                patient_data["target"], patient_data["y_pred"]
            )
            * 100
        )
        rmses.append(
            root_mean_squared_error(patient_data["target"], patient_data["y_pred"])
        )

    return samples, maes, mapes, rmses

In [9]:
def print_results(df):
    """Stampa i risultati delle valutazioni cumulative e per condizione glicemica.
    Args:
        df (pd.DataFrame): DataFrame con valori inferiti e di riferimento
    """

    def print_metrics(title, samples, maes, mapes, rmses):
        """Stampa le metriche formattate per una specifica condizione."""
        if title != "Cumulative":
            print("~" * 10)
        print(title)
        print(f"Samples: {samples}")
        if maes:  # Stampa solo se abbiamo dei dati
            print(f"MAE: {np.mean(maes):.2f}({np.std(maes):.2f})")
            print(f"MAPE: {np.mean(mapes):.2f}({np.std(mapes):.2f})")
            print(f"RMSE: {np.mean(rmses):.2f}({np.std(rmses):.2f})")

    # Overall results
    samples, maes, mapes, rmses = calculate_metrics(df)
    print_metrics("Cumulative", samples, maes, mapes, rmses)

    # Results by condition
    for condition in ["Normal", "Hyper", "Hypo"]:
        condition_df = df[df["bgClass"] == condition]
        samples, maes, mapes, rmses = calculate_metrics(condition_df)
        print_metrics(condition, samples, maes, mapes, rmses)

### Training Std_ML Utilities


In [10]:
def create_model(model_type, seed):
    """Costruisci il modello specificato"""
    if model_type == "lgb":
        return lgb.LGBMRegressor(random_state=seed, device="gpu")
    elif model_type == "xgb":
        return xgb.XGBRegressor(random_state=seed, device="cuda:0")
    elif model_type == "rf":
       return RandomForestRegressor(random_state=seed)
    else:
        raise ValueError(f"Unsupported model type: {model_type}")

In [11]:
def save_model(model, model_path):
    """Salva il modello addestrato"""
    with open(model_path, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"Model saved to: {model_path}")

In [12]:
def evaluate_and_save_results(model, val_set, X_cols, y_cols, output_path, exp_name):
    """Valuta e salva i risultati del modello"""
    # Inferisci
    val_set = val_set.copy()
    val_set["y_pred"] = model.predict(val_set[X_cols])

    # Prepara i risultati
    val_set = val_set.rename(columns={y_cols[-1]: "target"})
    val_set = rescale_data(val_set, ["target", "y_pred"])

    # Seleziona le colonne di output e salva
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = val_set[output_columns]

    print_results(results)

    output_file = f"{output_path}/{exp_name}_output.csv"
    results.to_csv(output_file, index=False)
    print(f"Results saved to: {output_file}")

    return results

### Main Pipeline


In [25]:
# Configura il training
class Args:
    def __init__(self):
        self.output_path = "outputs/val_set"
        self.models_path = "models/val_set"
        self.exp_name = "rf"  # Cambia qui: "lgb", "xgb", "rf"
        self.seed = 42


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - Model: {args.exp_name}")
print(f"  - Output path: {args.output_path}")
print(f"  - Models path: {args.models_path}")
print(f"  - Seed: {args.seed}")

In [26]:
print(f"Starting {args.exp_name.upper()} training pipeline...")

# Setup ambiente
os.makedirs(args.output_path, exist_ok=True)
os.makedirs(args.models_path, exist_ok=True)

In [27]:
# Loading dei set dati
print("Loading pre-prepared data splits...")
train_set, val_set, test_set, X_cols, y_cols = load_splits()

In [28]:
# Costruisci e addestra il modello
print(f"Creating and training {args.exp_name.upper()} model...")
model = create_model(args.exp_name, args.seed)
model.fit(train_set[X_cols], train_set[y_cols[-1]])

In [29]:
# Salva il modello
model_path = f"{args.models_path}/{args.exp_name}.pickle"
save_model(model, model_path)

In [30]:
# Valuta e salva il modello
results = evaluate_and_save_results(
    model, val_set, X_cols, y_cols, args.output_path, args.exp_name
)
print(f"\nTraining pipeline completed successfully!")